# 12 — Review grounding V2
Review saved training logs and eight validation previews in VS Code. No model loading, GPU, or network is needed.

Place this notebook in `agent1/notebooks/`. Copy the five JSON files into
`agent1/experiments/grounding_v2/run_20261024T220321_655458Z/`.
Select your Agent1 Python kernel, then run the cells in order.

This notebook compares the saved previews; it does **not** generate the remaining 40 validation responses or select a production adapter automatically.


In [1]:
from pathlib import Path
import json
from html import escape
from IPython.display import display, HTML, Markdown

# Find the Agent1 root relative to the current notebook/kernel directory.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "agent1").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the Agent1 project, which contains src/agent1.")

RUN_NAME = "run_20261024T220321_655458Z"  # Change only if your run folder has a different name.
run_dir = project_root / "experiments" / "grounding_v2" / RUN_NAME
steps = [0, 30, 60, 90]
paths = {s: run_dir / f"validation_preview_step_{s:04d}.json" for s in steps}
required = [run_dir / "trainer_log.json", *paths.values()]
missing = [str(p.relative_to(project_root)) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Copy these files into the project:\n" + "\n".join(missing))
log = json.loads(required[0].read_text(encoding="utf-8-sig"))
previews = {s: json.loads(p.read_text(encoding="utf-8-sig")) for s, p in paths.items()}
by_step = {s: {r["id"]: r for r in rows} for s, rows in previews.items()}
ids = list(by_step[0])
for s in steps:
    assert len(by_step[s]) == len(previews[s]), f"Duplicate IDs at step {s}"
    assert set(by_step[s]) == set(ids), f"Different examples at step {s}"
    for case_id in ids:
        row, base = by_step[s][case_id], by_step[0][case_id]
        assert row["prompt"] == base["prompt"], f"Changed prompt: {case_id}"
        assert row["target"] == base["target"], f"Changed target: {case_id}"
        assert row["prompt_type"] == base["prompt_type"]
print("Project root:", project_root)
print("Run:", run_dir)
print(f"Loaded {len(ids)} matching examples at each of {len(steps)} stages.")


Project root: d:\Projects\projects\agent1
Run: d:\Projects\projects\agent1\experiments\grounding_v2\run_20261024T220321_655458Z
Loaded 8 matching examples at each of 4 stages.


## Training metrics
Validation loss uses all 48 validation examples and their reference answers. Preview generation covers only eight examples.
Token accuracy is measured with preceding reference tokens supplied (teacher forcing); it is **not** an answer-level grounding accuracy.
Exact text matches below measure wording agreement, not semantic correctness. A valid paraphrase may differ from the target.

The uploaded trainer log has no step-0 evaluation entry. We leave that loss unavailable rather than infer it from the first training loss.


In [2]:
def table(headers, rows):
    head = "".join(f"<th>{escape(str(x))}</th>" for x in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(x))}</td>" for x in row) + "</tr>" for row in rows)
    display(HTML("<style> .eval12 td,.eval12 th {padding:8px;border:1px solid #bbb;text-align:left;vertical-align:top;white-space:pre-wrap} </style>"
                 + f"<div style='overflow-x:auto'><table class='eval12'><tr>{head}</tr>{body}</table></div>"))

eval_rows = {int(r["step"]): r for r in log if "eval_loss" in r}
summary = []
for s in steps:
    rows = previews[s]
    ev = eval_rows.get(s, {})
    item = {
        "step": s, "epoch": s / 30,
        "eval_loss": ev.get("eval_loss"),
        "eval_token_accuracy": ev.get("eval_mean_token_accuracy"),
        "preview_count": len(rows),
        "exact_text_matches": sum(r["answer"] == r["target"] for r in rows),
        "mean_words": sum(len(r["answer"].split()) for r in rows) / len(rows),
        "hit_token_limit": sum(bool(r["hit_token_limit"]) for r in rows),
    }
    summary.append(item)
table(["Step", "Epoch", "Validation loss", "Token accuracy", "Exact wording / 8", "Mean words", "Token limit hits"], [
    [r["step"], r["epoch"], f"{r['eval_loss']:.6f}" if r["eval_loss"] is not None else "not in log",
     f"{r['eval_token_accuracy']:.2%}" if r["eval_token_accuracy"] is not None else "not in log",
     r["exact_text_matches"], f"{r['mean_words']:.2f}", r["hit_token_limit"]] for r in summary
])
last = next((r for r in reversed(log) if "train_runtime" in r), {})
if last:
    print(f"Logged training runtime: {last['train_runtime'] / 60:.1f} minutes")
print("Identical epoch-2 and epoch-3 answers:", sum(by_step[60][i]["answer"] == by_step[90][i]["answer"] for i in ids), "/", len(ids))


Step,Epoch,Validation loss,Token accuracy,Exact wording / 8,Mean words,Token limit hits
0,0.0,not in log,not in log,1,12.38,0
30,1.0,0.106276,97.48%,5,6.88,0
60,2.0,0.003458,100.00%,8,8.50,0
90,3.0,0.002569,100.00%,8,8.50,0


Logged training runtime: 38.4 minutes
Identical epoch-2 and epoch-3 answers: 8 / 8


## Loss curves
Training points are averages over logging intervals; validation points use the validation set. They are different measurements. The log scale makes small late-stage losses visible. A falling validation loss on shared templates does not demonstrate robustness to new prompt styles.


In [3]:
# A self-contained SVG plot avoids requiring matplotlib in the local environment.
import math
train_points = [(r["step"], r["loss"]) for r in log if "loss" in r]
validation_points = [(r["step"], r["eval_loss"]) for r in log if "eval_loss" in r]
all_points = train_points + validation_points
assert all(v > 0 for _, v in all_points), "Log plot requires positive loss."
xmax = max(x for x, _ in all_points)
lo = math.floor(min(math.log10(v) for _, v in all_points))
hi = math.ceil(max(math.log10(v) for _, v in all_points))
def xy(step, loss):
    return 75 + step / xmax * 680, 290 - (math.log10(loss) - lo) / max(hi-lo, 1) * 240
parts = ["<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 820 360' role='img' aria-label='Training and validation loss by optimizer step' style='max-width:900px;background:white;color:#111'>"]
for exponent in range(lo, hi+1):
    y = xy(0, 10**exponent)[1]
    parts.append(f"<line x1='75' x2='755' y1='{y}' y2='{y}' stroke='#ddd'/><text x='10' y='{y+4}' font-size='12'>{10**exponent:g}</text>")
for s in steps:
    x = xy(s, 1)[0]
    parts.append(f"<text x='{x}' y='315' text-anchor='middle' font-size='12'>{s}</text>")
for label, points, color in [("Training", train_points, "#2463b5"), ("Validation", validation_points, "#b34700")]:
    coords = " ".join(f"{x},{y}" for x,y in [xy(s,v) for s,v in points])
    parts.append(f"<polyline points='{coords}' fill='none' stroke='{color}' stroke-width='2'/>")
    for s,v in points:
        x,y=xy(s,v)
        parts.append(f"<circle cx='{x}' cy='{y}' r='4' fill='{color}'><title>{label}: step {s}, loss {v:.6f}</title></circle>")
parts.extend(["<text x='75' y='22' fill='#2463b5'>Training</text><text x='175' y='22' fill='#b34700'>Validation</text>",
              "<text x='340' y='345'>Optimizer step</text><text x='560' y='22'>Loss (log scale)</text></svg>"])
display(HTML("".join(parts)))


## Compare the same prompt across epochs
For each case, read the supplied context and question first. Judge factual correctness independently of target wording.
The system prompt is available in the expandable section. No hidden context is omitted from the stored prompt.


In [4]:
for case_id in ids:
    base = by_step[0][case_id]
    display(Markdown(f"### {base['prompt_type']} — `{case_id}`"))
    system = "\n\n".join(m["content"] for m in base["prompt"] if m["role"] == "system")
    display(HTML("<details><summary>System instructions</summary><pre style='white-space:pre-wrap'>" + escape(system) + "</pre></details>"))
    for message in base["prompt"]:
        if message["role"] != "system":
            display(HTML("<pre style='white-space:pre-wrap'>" + escape(message["content"]) + "</pre>"))
    table(["Reference answer"], [[base["target"]]])
    table(["Before training", "Epoch 1", "Epoch 2", "Epoch 3"],
          [[by_step[s][case_id]["answer"] for s in steps]])


### action_confirmation — `validation-action_confirmation-10-2`

Reference answer
"Cancellation of ORD-2110 has not been performed, so I cannot confirm it is cancelled."


Before training,Epoch 1,Epoch 2,Epoch 3
I'm unable to cancel ORD-2110 as no cancellation has been attempted or completed.,I could not cancel ORD-2110.,"Cancellation of ORD-2110 has not been performed, so I cannot confirm it is cancelled.","Cancellation of ORD-2110 has not been performed, so I cannot confirm it is cancelled."


### direct_fact — `validation-direct_fact-11-0`

Reference answer
Contact support at help110@example.com.


...

Reference answer
Refund REF-3111 for CAD 46 is paid.


Before training,Epoch 1,Epoch 2,Epoch 3
"The refund status for REF-3111 is ""paid"".",Refund REF-3111 is paid for CAD 46.,Refund REF-3111 for CAD 46 is paid.,Refund REF-3111 for CAD 46 is paid.


### unavailable_fact — `validation-unavailable_fact-10-0`

Reference answer
Holiday opening hours for Alder are not available.


Before training,Epoch 1,Epoch 2,Epoch 3
"Unfortunately, I don't have any information on holiday hours for Alder. Can you please clarify what specific holidays you would like to know about?",Holiday opening hours for Alder are not available.,Holiday opening hours for Alder are not available.,Holiday opening hours for Alder are not available.


## Interpretation of this uploaded run
- **Missing branch:** step 0 picks Alder without justification. From epoch 1 onward the answer asks which branch is intended. This is a substantive grounding improvement.
- **Action not performed:** the epoch-1 answer “I could not cancel” can imply a failed attempt. Epochs 2 and 3 explicitly say cancellation was not performed, which matches the context.
- **Lookup failure:** the baseline mentions “tool result.” Epoch 1 removes this internal wording; epochs 2 and 3 also suggest checking the refund ID.
- **Unavailable holiday hours:** the baseline asks which holiday even though no holiday hours are provided. Tuned answers state the limitation directly.
- **Direct fact, partial information, scope and successful lookup:** the baseline already gives the core correct facts. Tuning mostly makes answers more concise and closer to the reference wording. The baseline refund answer correctly gives the requested status even without the amount.

Epochs 2 and 3 exactly match all eight targets and each other. This is not “100% grounding accuracy”: only eight generated previews were reviewed. The full-set validation token accuracy is also not a free-generation success rate.

Training and validation were generated from shared templates with different scenario groups. Very low loss can reflect learning those templates. There is no increase in logged validation loss, but these results alone cannot establish generalization or rule out template overfitting.

**Candidate shortlist:** checkpoints 60 and 90. Use full validation generation to compare them before choosing. Epoch 3 has lower loss, but no visible preview advantage. Do not train longer merely to reduce this already small loss.


## Record your review
Use `pass`, `partial`, or `fail` for each criterion. Leave a field as `None` until you have reviewed it. For **unsupported_claims**, `pass` means no unsupported claim is present. Exact match does not assign these ratings.

`context_use`: correct facts, scope, and action state.  
`clarification`: asks only when useful and does not ask for known information.  
`unsupported_claims`: no invented facts, procedures, or actions.  
`relevance_brevity`: directly relevant, without unnecessary filler.

Edit the `ratings` dictionary below if you want to save your judgments. The optional export writes a new timestamped JSON file and preserves the original logs.


In [5]:
criteria = ["context_use", "clarification", "unsupported_claims", "relevance_brevity"]
ratings = {}  # Keys are (step, case_id).
# Example after reviewing the missing-branch answer yourself:
# ratings[(60, "validation-missing_identifier-10-1")] = {
#     "context_use": "pass", "clarification": "pass",
#     "unsupported_claims": "pass", "relevance_brevity": "pass",
#     "notes": "Asks which branch instead of choosing one."
# }
review_rows = []
for s in steps:
    for case_id in ids:
        row = by_step[s][case_id]
        rating = ratings.get((s, case_id), {})
        assert all(rating.get(k) in (None, "pass", "partial", "fail") for k in criteria)
        review_rows.append({
            "step": s, "id": case_id, "prompt_type": row["prompt_type"],
            "answer": row["answer"], "target": row["target"],
            **{k: rating.get(k) for k in criteria}, "notes": rating.get("notes", ""),
        })
SAVE_REVIEW = False  # Set True only when you want to export your review.
if SAVE_REVIEW:
    from datetime import datetime, timezone
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    out = run_dir / f"preview_review_{stamp}.json"
    out.write_text(json.dumps({"summary": summary, "review": review_rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", out)
else:
    print("Review prepared in memory. Original results are unchanged.")


Review prepared in memory. Original results are unchanged.


## Next evaluation step
Keep `checkpoints/checkpoint-60` and `checkpoints/checkpoint-90` in the Colab run folder. Generate responses for **all 48 validation prompts** with each checkpoint, using identical decoding settings and the same saved prompt snapshot. Compare with the same rubric above, by prompt class. This notebook cannot reconstruct missing responses from loss values.

After selecting a candidate, copy its adapter configuration and weights, plus the tokenizer files from the run, into `models/adapter_v2/` in Agent1. Keep V1 separate. Run the existing 24-case grounding benchmark against the selected V2 adapter with the same prompts and generation settings used for the recorded base results.

That benchmark is an existing diagnostic already used to guide V2, so it is not an untouched final test. Add an independently written test set with new phrasing, changed values, reordered/distracting facts, and missing/conflicting information to assess generalization. Do not add evaluation examples to training while treating them as held out.
